# Data Mining Final Project: Computer Science Journal Finder

This notebook is the main reproducible workflow for the project. It is written as a step-by-step explanation so that the implementation can be followed without reading the source code first.

The project has two required software outputs:

1. Given an article abstract, recommend the **top 5 most relevant computer science journals**.
2. Generate **topic clusters** from the publication database.

The implementation uses transparent data mining methods: text preprocessing, TF-IDF vectorization, cosine similarity, journal-level score aggregation, and KMeans clustering.


## 0. Assignment Requirement Mapping

The table below maps the PDF requirements directly to this project. This is included first so the notebook is easy to grade against the assignment document.

| PDF requirement | Where it is handled in this project |
|---|---|
| Source code with GitHub link | `src/`, `app.py`, and `README.md` |
| Jupyter Notebook format | This notebook |
| IEEE Conference report with literature review | `report/ieee_report.tex`, `report/ieee_report.md`, `report/ieee_report.docx` |
| Journal finder software tailored to computer science subject areas | Streamlit app in `app.py`; recommender in `src/recommender.py` |
| Generate clusters of topics for subject areas | `cluster_topics` in `src/modeling.py` and Section 11 of this notebook |
| Author enters article abstract and program lists top 5 relevant journals | `recommend_journals(..., top_n=5)` and Sections 7-10 of this notebook |

Important note: the PDF does **not** define a minimum abstract length. Therefore, the software rejects only empty input.


## 1. Method Overview

The complete pipeline is:

1. Load article records from the SQLite database.
2. Join article metadata with abstracts, journals, author keywords, Web of Science keyword plus terms, and subject labels.
3. Clean text by removing HTML, normalizing whitespace, and lowercasing.
4. Build one weighted training document per article.
5. Train a TF-IDF vectorizer over all article documents.
6. Convert the user abstract into the same TF-IDF space.
7. Use cosine similarity to find related articles.
8. Aggregate article similarities by journal.
9. Return the top 5 journals.
10. Cluster all article vectors with KMeans for topic discovery.

This approach is intentionally explainable: each recommended journal can be traced back to similar articles in the database.


## 2. Load Libraries and Project Modules

This cell imports the reusable project modules. The path logic allows the notebook to run whether Jupyter is opened from the project root or directly from the `notebooks/` folder.


In [1]:
from pathlib import Path
import sys
import sqlite3
import subprocess

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_dataset, summarize_dataset
from src.modeling import cluster_topics, train_vectorizer
from src.recommender import format_recommendations, recommend_journals
from src.text_preprocessing import clean_text

DB_PATH = PROJECT_ROOT / 'CompSciencePub.sqlite'
EXPORTS_DIR = PROJECT_ROOT / 'exports'
EXPORTS_DIR.mkdir(exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Database path:', DB_PATH)
print('Database exists:', DB_PATH.exists())


Project root: C:\Users\umut3\Downloads\DATA_MINING
Database path: C:\Users\umut3\Downloads\DATA_MINING\CompSciencePub.sqlite
Database exists: True


## 3. Inspect the Raw Database Structure

Before building a model, it is useful to verify what the database contains. The assignment PDF shows the important entities, and this cell confirms the tables and row counts from the actual SQLite file.

The most important tables for this project are:

- `AcademicRecord`: article metadata and publication id.
- `AcademicRecordAbstract`: article abstract text.
- `Publication`: journal names.
- `AcademicRecordKeyword` and `AcademicKeyword`: author keywords.
- `AcademicRecordKeywordPlus` and `AcademicKeywordPlus`: Web of Science keyword plus terms.
- `AcademicRecordSubject` and `AcademicSubject`: subject categories.


In [2]:
with sqlite3.connect(DB_PATH) as connection:
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
        connection,
    )
    table_summary = []
    for table_name in tables['name']:
        row_count = connection.execute(f'SELECT COUNT(*) FROM "{table_name}"').fetchone()[0]
        table_summary.append({'table': table_name, 'rows': row_count})

pd.DataFrame(table_summary).sort_values('rows', ascending=False).reset_index(drop=True)


,table,rows
0,AcademicRecordContributorOrganizationSet,138860
1,AcademicRecordKeyword,101769
2,AcademicRecordSubject,86033
3,AcademicRecordContributor,84865
4,AcademicRecordKeywordPlus,83797
5,AcademicContributor,76343
6,AcademicRecordContributorSubOrganizationSet,62980
7,AcademicKeyword,52058
8,AcademicRecordSubHeading,27340
9,AcademicRecordHeading,24819


## 4. Load the Modeling Dataset

`load_dataset` performs the SQL joins and returns one row per usable article abstract. Records without usable abstracts or journals are removed because they cannot support abstract-to-journal recommendation.


In [3]:
df = load_dataset(DB_PATH)
summary = summarize_dataset(df)

pd.DataFrame([summary])


,articles,journals,subjects
0,23061,455,80


## 5. Preview the Article-Level Data

The recommender is trained from article records. The preview below shows the fields a human would also inspect when deciding whether a journal fits an abstract: title, journal, year, subjects, and keyword fields.


In [4]:
preview_columns = [
    'record_id',
    'title',
    'journal',
    'pub_year',
    'author_keywords',
    'keyword_plus',
    'subjects',
]
df[preview_columns].head(10)


,record_id,title,journal,pub_year,author_keywords,keyword_plus,subjects
0,88652,An updated survey of GA-based multiobjective o...,ACM COMPUTING SURVEYS,2000,"algorithms, artificial intelligence, genetic a...","GENETIC ALGORITHM, MULTICRITERIA OPTIMIZATION,...","Computer Science, Theory & Methods, Computer S..."
1,88653,The state of the art in distributed query proc...,ACM COMPUTING SURVEYS,2000,"query optimization, query execution, client-se...","DATABASE-SYSTEMS, DATA REPLICATION, PERFORMANC...","Computer Science, Theory & Methods, Computer S..."
2,88654,Logical models of argument,ACM COMPUTING SURVEYS,2000,"defeasible argumentation, argumentative system...","IMPLEMENTATION, FRAMEWORK","Computer Science, Theory & Methods, Computer S..."
3,88655,Information retrieval on the Web,ACM COMPUTING SURVEYS,2000,"algorithms, theory, clustering, indexing, info...","WORLD-WIDE-WEB, SEARCH, SYSTEM, DIRECTIONS, EN...","Computer Science, Theory & Methods, Computer S..."
4,88656,A guided tour to approximate string matching,ACM COMPUTING SURVEYS,2001,"algorithms, edit distance, Levenshtein distanc...","FAST ALGORITHMS, SUBQUADRATIC ALGORITHM, COMMO...","Computer Science, Theory & Methods, Computer S..."
5,88657,Searching in metric spaces,ACM COMPUTING SURVEYS,2001,"algorithms, curse of dimensionality, nearest n...","NEAREST-NEIGHBOR SEARCH, TREES, FILE, ALGORITH...","Computer Science, Theory & Methods, Computer S..."
6,88658,Searching in high-dimensional spaces - Index s...,ACM COMPUTING SURVEYS,2001,"algorithms, design, measurement, performance, ...","GRID FILE, TREES, DESCRIPTORS","Computer Science, Theory & Methods, Computer S..."
7,88659,Complexity and expressive power of logic progr...,ACM COMPUTING SURVEYS,2001,"languages, theory, complexity, datalog, expres...","WELL-FOUNDED SEMANTICS, STABLE MODEL SEMANTICS...","Computer Science, Theory & Methods, Computer S..."
8,88660,Machine learning in automated text categorization,ACM COMPUTING SURVEYS,2002,"algorithms, experimentation, theory, machine l...","PROBABILISTIC INFORMATION-RETRIEVAL, DOCUMENT ...","Computer Science, Theory & Methods, Computer S..."
9,88661,A survey of rollback-recovery protocols in mes...,ACM COMPUTING SURVEYS,2002,"design, reliability, performance, message logg...","PREVENTING USELESS CHECKPOINTS, DISTRIBUTED SY...","Computer Science, Theory & Methods, Computer S..."


## 6. Text Cleaning Example

The database abstracts may contain HTML tags and encoded characters. The cleaning step converts them into plain searchable text.


In [5]:
raw_example = df.loc[0, 'abstract']
cleaned_example = clean_text(raw_example)

print('Raw abstract preview:')
print(raw_example[:500])
print()
print('Cleaned abstract preview:')
print(cleaned_example[:500])


Raw abstract preview:
<p>After using evolutionary techniques for single-objective optimization during more than two decades, the incorporation of more than one objective in the fitness function has finally become a popular area of research. As a consequence, many new evolutionary-based approaches and variations of existing techniques have recently been published in the technical literature. The purpose of this paper is to summarize and organize the information on these current approaches, emphasizing the importance o

Cleaned abstract preview:
after using evolutionary techniques for single-objective optimization during more than two decades the incorporation of more than one objective in the fitness function has finally become a popular area of research as a consequence many new evolutionary-based approaches and variations of existing techniques have recently been published in the technical literature the purpose of this paper is to summarize and organize the information on these curre

## 7. Final Training Text Representation

Each article is represented by a combined text field called `training_text`.

The optimized representation uses:

- title
- title again, to strengthen article scope
- abstract
- author keywords
- author keywords again, to strengthen author-provided topic labels
- Web of Science keyword plus terms
- subject labels
- subject labels again, to strengthen computer science subject area signals

This weighting improves journal matching because journals are often defined by scope and subject category, not only by abstract wording.


In [6]:
text_columns = [
    'clean_title',
    'clean_abstract',
    'clean_author_keywords',
    'clean_keyword_plus',
    'clean_subjects',
    'training_text',
]
df.loc[:2, text_columns]


,clean_title,clean_abstract,clean_author_keywords,clean_keyword_plus,clean_subjects,training_text
0,an updated survey of ga-based multiobjective o...,after using evolutionary techniques for single...,algorithms artificial intelligence genetic alg...,genetic algorithm multicriteria optimization s...,computer science theory methods computer science,an updated survey of ga-based multiobjective o...
1,the state of the art in distributed query proc...,distributed data processing is becoming a real...,query optimization query execution client-serv...,database-systems data replication performance ...,computer science theory methods computer science,the state of the art in distributed query proc...
2,logical models of argument,logical models of argument formalize commonsen...,defeasible argumentation argumentative systems...,implementation framework,computer science theory methods computer science,logical models of argument logical models of a...


## 8. Train the TF-IDF Representation

TF-IDF converts every article into a numeric vector.

Current vectorizer settings:

- English stop words removed.
- Unigrams and bigrams are used.
- `min_df=1` keeps rare technical terms.
- `max_df=0.9` removes terms that appear in almost all documents.
- `max_features=80000` keeps the strongest vocabulary terms.
- `sublinear_tf=True` reduces the effect of repeated words.
- L2 normalization makes cosine similarity meaningful.


In [7]:
vectorizer, matrix = train_vectorizer(df['training_text'])
features = vectorizer.get_feature_names_out()

print('TF-IDF matrix shape:', matrix.shape)
print('Number of features:', len(features))
print('First 20 features:', features[:20].tolist())


TF-IDF matrix shape: (23061, 80000)
Number of features: 80000
First 20 features: ['00', '000', '000 000', '000 lines', '000 users', '009', '01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '10 000', '10 10', '10 1002', '10 1057']


## 9. Recommendation Scoring Logic

For a user abstract, the recommender:

1. Cleans the input abstract.
2. Converts it into the trained TF-IDF space.
3. Computes cosine similarity against all known article records.
4. Keeps the strongest neighboring articles.
5. Groups neighbors by journal.
6. Scores each journal using:

`journal_score = 0.60 * best_article_similarity + 0.25 * mean_article_similarity + 0.15 * evidence_score`

The evidence score rewards journals that have multiple relevant matched articles, not only one isolated similar article.


## 10. Main Example: Top-5 Journals for One Abstract

This cell demonstrates the main assignment requirement. The input is a wireless sensor network abstract, and the output is the top 5 relevant journals.


In [8]:
sample_abstract = """
This paper investigates energy efficient routing protocols for wireless sensor networks.
The proposed algorithm reduces packet loss and improves network lifetime by optimizing
cluster head selection and transmission paths. Simulation results evaluate throughput,
delay, energy consumption, and scalability under different sensor deployment scenarios.
"""

recommendations = format_recommendations(
    recommend_journals(sample_abstract, df, vectorizer, matrix, top_n=5)
)
recommendations


,rank,journal,score,matched_articles,example_title,subjects
0,1,INFORMATION SYSTEMS FRONTIERS,0.2900,1,T-LEACH: The method of threshold-based cluster...,"Computer Science, Information Systems, Compute..."
1,2,AD HOC & SENSOR WIRELESS NETWORKS,0.2253,11,Improvement of the Wireless Sensor Network Lif...,"Computer Science, Information Systems, Telecom..."
2,3,WIRELESS NETWORKS,0.2240,20,A particle swarm optimization based energy eff...,"Computer Science, Information Systems, Enginee..."
3,4,COMPUTER SYSTEMS SCIENCE AND ENGINEERING,0.2180,6,Routing protocols for mobile sensor networks: ...,"Computer Science, Hardware & Architecture, Com..."
4,5,INTERNATIONAL JOURNAL OF SENSOR NETWORKS,0.2117,10,Clustering algorithm for wireless sensor netwo...,"Computer Science, Information Systems, Telecom..."


## 11. Interpreting Recommendation Output

The output columns mean:

- `rank`: final position in the top-5 list.
- `journal`: recommended journal name.
- `score`: final aggregated relevance score.
- `matched_articles`: number of similar neighboring articles from that journal.
- `example_title`: one matched article title used as evidence.
- `subjects`: subject categories attached to the matched article.

The example title and subjects make the result explainable, which is useful during presentation.


In [9]:
recommendations[['rank', 'journal', 'score', 'matched_articles', 'example_title', 'subjects']]


,rank,journal,score,matched_articles,example_title,subjects
0,1,INFORMATION SYSTEMS FRONTIERS,0.2900,1,T-LEACH: The method of threshold-based cluster...,"Computer Science, Information Systems, Compute..."
1,2,AD HOC & SENSOR WIRELESS NETWORKS,0.2253,11,Improvement of the Wireless Sensor Network Lif...,"Computer Science, Information Systems, Telecom..."
2,3,WIRELESS NETWORKS,0.2240,20,A particle swarm optimization based energy eff...,"Computer Science, Information Systems, Enginee..."
3,4,COMPUTER SYSTEMS SCIENCE AND ENGINEERING,0.2180,6,Routing protocols for mobile sensor networks: ...,"Computer Science, Hardware & Architecture, Com..."
4,5,INTERNATIONAL JOURNAL OF SENSOR NETWORKS,0.2117,10,Clustering algorithm for wireless sensor netwo...,"Computer Science, Information Systems, Telecom..."


## 12. Empty and Short Input Behavior

The PDF only says the author enters an article abstract. It does not say there must be at least 20 words. Therefore:

- Empty input is rejected.
- Short non-empty text still returns top-5 recommendations.

This is closer to the assignment wording than enforcing an artificial word limit.


In [10]:
short_input = 'wireless sensor network routing energy'
short_result = format_recommendations(
    recommend_journals(short_input, df, vectorizer, matrix, top_n=5)
)
short_result


,rank,journal,score,matched_articles,example_title,subjects
0,1,WIRELESS NETWORKS,0.2659,16,A reliable energy-efficient pressure-based rou...,"Computer Science, Information Systems, Enginee..."
1,2,JOURNAL OF SUPERCOMPUTING,0.2577,4,An efficient routing algorithm to preserve -co...,"Computer Science, Hardware & Architecture, Com..."
2,3,TELECOMMUNICATION SYSTEMS,0.2523,3,Routing in mobile wireless sensor network: a s...,"Telecommunications, Telecommunications"
3,4,INTERNATIONAL JOURNAL OF DISTRIBUTED SENSOR NE...,0.2361,14,An adaptive data dissemination strategy for wi...,"Computer Science, Information Systems, Telecom..."
4,5,AD HOC & SENSOR WIRELESS NETWORKS,0.2303,17,Macroscopic Analysis of Wireless Sensor Networ...,"Computer Science, Information Systems, Telecom..."


In [11]:
try:
    recommend_journals('', df, vectorizer, matrix, top_n=5)
except ValueError as error:
    print('Empty input validation message:', error)


Empty input validation message: Please enter an article abstract.


## 13. Ten Test Abstracts

The following test abstracts cover multiple computer science subfields. They are used as a practical evaluation set for checking whether the recommender behaves sensibly across different topics.


In [12]:
test_abstracts = [
    {
        'id': 1,
        'topic': 'Wireless sensor networks',
        'abstract': 'This paper investigates energy efficient routing protocols for wireless sensor networks. The proposed algorithm reduces packet loss and improves network lifetime by optimizing cluster head selection and transmission paths. Simulation results evaluate throughput, delay, energy consumption, and scalability under different sensor deployment scenarios.',
    },
    {
        'id': 2,
        'topic': 'Machine learning classification',
        'abstract': 'This study proposes a machine learning framework for supervised classification on high dimensional datasets. Feature selection, support vector machines, random forests, and cross validation are used to improve predictive accuracy. Experimental results compare precision, recall, F measure, and robustness across benchmark datasets with noisy attributes.',
    },
    {
        'id': 3,
        'topic': 'Image processing and computer vision',
        'abstract': 'This paper presents an image segmentation and object recognition method for complex visual scenes. The approach combines texture descriptors, edge information, and convolutional feature representations to improve detection accuracy. Experiments on image datasets evaluate segmentation quality, recognition rate, computational cost, and robustness to illumination changes.',
    },
    {
        'id': 4,
        'topic': 'Cryptography and network security',
        'abstract': 'This research develops a secure authentication and encryption protocol for distributed network communication. The method uses lightweight cryptographic primitives, key agreement, and intrusion detection rules to protect confidentiality and integrity. Security analysis and performance experiments measure resistance to attacks, computational overhead, and communication latency.',
    },
    {
        'id': 5,
        'topic': 'Cloud computing scheduling',
        'abstract': 'This paper proposes a resource allocation and task scheduling algorithm for cloud computing environments. Virtual machines are assigned dynamically according to workload, response time, energy consumption, and service level constraints. Simulation experiments evaluate makespan, throughput, load balancing, and scalability under heterogeneous data center workloads.',
    },
    {
        'id': 6,
        'topic': 'Software engineering testing',
        'abstract': 'This study introduces an automated software testing approach for fault detection and regression test prioritization. Static code metrics, execution traces, and historical defect data are analyzed to rank test cases. Empirical evaluation on open source software projects measures coverage, fault detection rate, execution time, and maintainability impact.',
    },
    {
        'id': 7,
        'topic': 'Database query optimization',
        'abstract': 'This paper investigates query optimization techniques for large scale relational database systems. The proposed optimizer estimates cost using statistics, indexes, join ordering, and materialized views. Experimental results on transaction and analytical workloads evaluate query response time, execution plans, storage overhead, and scalability with increasing data volume.',
    },
    {
        'id': 8,
        'topic': 'Natural language processing',
        'abstract': 'This research presents a natural language processing model for text classification and semantic similarity detection. The method combines tokenization, term weighting, syntactic features, and neural word representations. Experiments on document collections evaluate accuracy, macro F score, topic consistency, and performance for short and long text inputs.',
    },
    {
        'id': 9,
        'topic': 'Human computer interaction',
        'abstract': 'This paper evaluates a human computer interaction interface designed to improve usability and user satisfaction in interactive systems. User studies measure task completion time, error rate, cognitive workload, accessibility, and perceived usefulness. The results show how interface design decisions affect learnability, efficiency, and user experience.',
    },
    {
        'id': 10,
        'topic': 'Data mining clustering',
        'abstract': 'This study proposes a data mining method for clustering large multidimensional datasets. The algorithm combines distance based similarity, dimensionality reduction, and density estimation to discover hidden patterns. Experiments compare cluster purity, silhouette score, runtime, noise sensitivity, and interpretability on benchmark and real world data collections.',
    },
]

test_abstract_table = pd.DataFrame(test_abstracts)
test_abstract_table


,id,topic,abstract
0,1,Wireless sensor networks,This paper investigates energy efficient routi...
1,2,Machine learning classification,This study proposes a machine learning framewo...
2,3,Image processing and computer vision,This paper presents an image segmentation and ...
3,4,Cryptography and network security,This research develops a secure authentication...
4,5,Cloud computing scheduling,This paper proposes a resource allocation and ...
5,6,Software engineering testing,This study introduces an automated software te...
6,7,Database query optimization,This paper investigates query optimization tec...
7,8,Natural language processing,This research presents a natural language proc...
8,9,Human computer interaction,This paper evaluates a human computer interact...
9,10,Data mining clustering,This study proposes a data mining method for c...


## 14. Run the Top-5 Recommendation Test Set

This section runs all 10 test abstracts through the same recommender. The first output contains every top-5 row; the second output shows only the first-ranked journal per topic.


In [13]:
all_recommendations = []

for item in test_abstracts:
    result = format_recommendations(
        recommend_journals(item['abstract'], df, vectorizer, matrix, top_n=5)
    )
    result.insert(0, 'topic', item['topic'])
    result.insert(0, 'case_id', item['id'])
    all_recommendations.append(result)

recommendation_results = pd.concat(all_recommendations, ignore_index=True)
recommendation_results


,case_id,topic,rank,journal,score,matched_articles,example_title,subjects
0,1,Wireless sensor networks,1,INFORMATION SYSTEMS FRONTIERS,0.2900,1,T-LEACH: The method of threshold-based cluster...,"Computer Science, Information Systems, Compute..."
1,1,Wireless sensor networks,2,AD HOC & SENSOR WIRELESS NETWORKS,0.2253,11,Improvement of the Wireless Sensor Network Lif...,"Computer Science, Information Systems, Telecom..."
2,1,Wireless sensor networks,3,WIRELESS NETWORKS,0.2240,20,A particle swarm optimization based energy eff...,"Computer Science, Information Systems, Enginee..."
3,1,Wireless sensor networks,4,COMPUTER SYSTEMS SCIENCE AND ENGINEERING,0.2180,6,Routing protocols for mobile sensor networks: ...,"Computer Science, Hardware & Architecture, Com..."
4,1,Wireless sensor networks,5,INTERNATIONAL JOURNAL OF SENSOR NETWORKS,0.2117,10,Clustering algorithm for wireless sensor netwo...,"Computer Science, Information Systems, Telecom..."
5,2,Machine learning classification,1,JOURNAL OF INTELLIGENT INFORMATION SYSTEMS,0.1828,5,Semi-supervised classification trees,"Computer Science, Artificial Intelligence, Com..."
6,2,Machine learning classification,2,ENVIRONMENTAL MODELLING & SOFTWARE,0.1653,1,A new synergistic approach for monitoring wetl...,"Computer Science, Interdisciplinary Applicatio..."
7,2,Machine learning classification,3,INTERNATIONAL JOURNAL OF DATA MINING AND BIOIN...,0.1632,8,A novel random forests-based feature selection...,"Mathematical & Computational Biology, Mathemat..."
8,2,Machine learning classification,4,KNOWLEDGE AND INFORMATION SYSTEMS,0.1546,9,A review of feature selection methods on synth...,"Computer Science, Artificial Intelligence, Com..."
9,2,Machine learning classification,5,COMPUTERS & GEOSCIENCES,0.1541,2,Geological mapping using remote sensing data: ...,"Computer Science, Interdisciplinary Applicatio..."


In [14]:
top_1_results = recommendation_results[recommendation_results['rank'] == 1][
    ['case_id', 'topic', 'journal', 'score', 'matched_articles', 'example_title']
]
top_1_results


,case_id,topic,journal,score,matched_articles,example_title
0,1,Wireless sensor networks,INFORMATION SYSTEMS FRONTIERS,0.2900,1,T-LEACH: The method of threshold-based cluster...
5,2,Machine learning classification,JOURNAL OF INTELLIGENT INFORMATION SYSTEMS,0.1828,5,Semi-supervised classification trees
10,3,Image processing and computer vision,JOURNAL OF VISUAL COMMUNICATION AND IMAGE REPR...,0.1477,9,Comparative study of global color and texture ...
15,4,Cryptography and network security,JOURNAL OF NETWORK AND COMPUTER APPLICATIONS,0.1330,13,A survey of intrusion detection techniques in ...
20,5,Cloud computing scheduling,CHINA COMMUNICATIONS,0.2305,1,Dynamic and Integrated Load-Balancing Scheduli...
25,6,Software engineering testing,IEEE TRANSACTIONS ON SOFTWARE ENGINEERING,0.1950,26,Prioritizing test cases for regression testing
30,7,Database query optimization,JOURNAL OF INTELLIGENT INFORMATION SYSTEMS,0.1912,4,Data mining-based materialized view and index ...
35,8,Natural language processing,DATA & KNOWLEDGE ENGINEERING,0.1891,7,SyMSS: A syntax-based measure for short-text s...
40,9,Human computer interaction,JOURNAL OF COMPUTER INFORMATION SYSTEMS,0.1504,4,Measuring user satisfaction and perceived usef...
45,10,Data mining clustering,INTELLIGENT DATA ANALYSIS,0.1351,7,Clustering of large time series datasets


## 15. Result Interpretation

The top-1 results show that the model is not returning one generic journal for every input. Different topics lead to different journal families:

- wireless networks return networking/systems journals,
- software testing returns software engineering journals,
- NLP returns data and knowledge engineering style journals,
- clustering returns data analysis journals.

This supports the goal of a computer science journal finder rather than a simple keyword search demo.


In [15]:
summary_for_presentation = top_1_results.copy()
summary_for_presentation['score'] = summary_for_presentation['score'].map(lambda value: round(float(value), 4))
summary_for_presentation


,case_id,topic,journal,score,matched_articles,example_title
0,1,Wireless sensor networks,INFORMATION SYSTEMS FRONTIERS,0.2900,1,T-LEACH: The method of threshold-based cluster...
5,2,Machine learning classification,JOURNAL OF INTELLIGENT INFORMATION SYSTEMS,0.1828,5,Semi-supervised classification trees
10,3,Image processing and computer vision,JOURNAL OF VISUAL COMMUNICATION AND IMAGE REPR...,0.1477,9,Comparative study of global color and texture ...
15,4,Cryptography and network security,JOURNAL OF NETWORK AND COMPUTER APPLICATIONS,0.1330,13,A survey of intrusion detection techniques in ...
20,5,Cloud computing scheduling,CHINA COMMUNICATIONS,0.2305,1,Dynamic and Integrated Load-Balancing Scheduli...
25,6,Software engineering testing,IEEE TRANSACTIONS ON SOFTWARE ENGINEERING,0.1950,26,Prioritizing test cases for regression testing
30,7,Database query optimization,JOURNAL OF INTELLIGENT INFORMATION SYSTEMS,0.1912,4,Data mining-based materialized view and index ...
35,8,Natural language processing,DATA & KNOWLEDGE ENGINEERING,0.1891,7,SyMSS: A syntax-based measure for short-text s...
40,9,Human computer interaction,JOURNAL OF COMPUTER INFORMATION SYSTEMS,0.1504,4,Measuring user satisfaction and perceived usef...
45,10,Data mining clustering,INTELLIGENT DATA ANALYSIS,0.1351,7,Clustering of large time series datasets


## 16. Topic Clustering Requirement

The assignment also asks for topic clusters for subject areas. This project uses KMeans over the article TF-IDF matrix. Each cluster is summarized using:

- `article_count`: number of articles in the cluster,
- `top_terms`: strongest centroid terms,
- `dominant_subjects`: most frequent subject labels,
- `sample_journals`: journals commonly appearing in the cluster.


In [16]:
clusters = cluster_topics(df, n_clusters=8)
clusters


,cluster,article_count,top_terms,dominant_subjects,sample_journals
0,1,8389,"software, software engineering, science softwa...","Computer Science, Software Engineering, Inform...",ACM TRANSACTIONS ON DESIGN AUTOMATION OF ELECT...
1,4,4433,"artificial, intelligence, artificial intellige...","Computer Science, Artificial Intelligence, The...","ARTIFICIAL INTELLIGENCE, ARTIFICIAL INTELLIGEN..."
2,7,2736,"interdisciplinary applications, interdisciplin...","Computer Science, Interdisciplinary Applicatio...",CMES-COMPUTER MODELING IN ENGINEERING & SCIENC...
3,2,2656,"engineering electrical, electrical electronic,...","Engineering, Computer Science, Telecommunicati...","DISPLAYS, IEEE JOURNAL ON SELECTED AREAS IN CO..."
4,6,2019,"telecommunications, telecommunications compute...","Computer Science, Telecommunications, Informat...","PHOTONIC NETWORK COMMUNICATIONS, MOBILE NETWOR..."
5,5,1501,"mathematics, science mathematics, mathematics ...","Computer Science, Mathematics, Applied, Theory...","COMPUTER AIDED GEOMETRIC DESIGN, DESIGNS CODES..."
6,3,788,"biology, computational biology, mathematical c...","Mathematical & Computational Biology, Computer...","COMPUTERS IN BIOLOGY AND MEDICINE, JOURNAL OF ..."
7,0,539,"science library, library science, information ...","Computer Science, Information Science & Librar...","JOURNAL OF MANAGEMENT INFORMATION SYSTEMS, ONL..."


## 17. Interpreting Clusters

A good cluster summary should contain coherent terms and subject labels. The table below keeps only the most presentation-friendly columns for quick review.


In [17]:
clusters[['cluster', 'article_count', 'top_terms', 'dominant_subjects']]


,cluster,article_count,top_terms,dominant_subjects
0,1,8389,"software, software engineering, science softwa...","Computer Science, Software Engineering, Inform..."
1,4,4433,"artificial, intelligence, artificial intellige...","Computer Science, Artificial Intelligence, The..."
2,7,2736,"interdisciplinary applications, interdisciplin...","Computer Science, Interdisciplinary Applicatio..."
3,2,2656,"engineering electrical, electrical electronic,...","Engineering, Computer Science, Telecommunicati..."
4,6,2019,"telecommunications, telecommunications compute...","Computer Science, Telecommunications, Informat..."
5,5,1501,"mathematics, science mathematics, mathematics ...","Computer Science, Mathematics, Applied, Theory..."
6,3,788,"biology, computational biology, mathematical c...","Mathematical & Computational Biology, Computer..."
7,0,539,"science library, library science, information ...","Computer Science, Information Science & Librar..."


## 18. Save Outputs for Report and Presentation

The notebook writes important outputs to `exports/`. These CSV files can be opened in Excel or copied into the report/presentation.


In [18]:
recommendation_path = EXPORTS_DIR / 'notebook_recommendation_results.csv'
top_1_path = EXPORTS_DIR / 'notebook_top_1_results.csv'
cluster_path = EXPORTS_DIR / 'notebook_topic_clusters.csv'
test_abstract_path = EXPORTS_DIR / 'notebook_test_abstracts.csv'

recommendation_results.to_csv(recommendation_path, index=False, encoding='utf-8-sig')
top_1_results.to_csv(top_1_path, index=False, encoding='utf-8-sig')
clusters.to_csv(cluster_path, index=False, encoding='utf-8-sig')
test_abstract_table.to_csv(test_abstract_path, index=False, encoding='utf-8-sig')

for path in [recommendation_path, top_1_path, cluster_path, test_abstract_path]:
    print(path)


C:\Users\umut3\Downloads\DATA_MINING\exports\notebook_recommendation_results.csv
C:\Users\umut3\Downloads\DATA_MINING\exports\notebook_top_1_results.csv
C:\Users\umut3\Downloads\DATA_MINING\exports\notebook_topic_clusters.csv
C:\Users\umut3\Downloads\DATA_MINING\exports\notebook_test_abstracts.csv


## 19. Automated Project Tests

The final verification step runs the unit test suite from inside the notebook. This confirms that the reusable code modules still work after changes.


In [19]:
completed = subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', str(PROJECT_ROOT / 'tests')],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
print(completed.stdout)
print(completed.stderr)
print('Return code:', completed.returncode)
assert completed.returncode == 0



.......
----------------------------------------------------------------------
Ran 7 tests in 3.228s

OK

Return code: 0


## 20. Final Checklist

The notebook demonstrates all core project requirements:

| Requirement | Demonstrated? |
|---|---|
| Load provided publication database | Yes |
| Use article abstracts and journal metadata | Yes |
| Use computer science subject information | Yes |
| Recommend top 5 journals for an entered abstract | Yes |
| Generate topic clusters | Yes |
| Save reproducible outputs | Yes |
| Run automated tests | Yes |

The same logic is available as reusable Python modules and through the Streamlit interface.
